In [1]:
import biom
import numpy as np
import pandas as pd
import subprocess
import os

table = biom.load_table('emp_deblur_150bp.release1.biom')
all_sample_ids = list(table.ids(axis='sample'))

batch_size = 100  # campioni per batch — riduci a 50 se continua a crashare
batches = [all_sample_ids[i:i+batch_size] for i in range(0, len(all_sample_ids), batch_size)]
print(f'Totale batch: {len(batches)} da {batch_size} campioni ciascuno')

results = []

for idx, batch_ids in enumerate(batches):
    print(f'Batch {idx+1}/{len(batches)}...', end=' ')
    
    # Filtra ai campioni del batch
    table_batch = table.filter(batch_ids, axis='sample', inplace=False)
    table_batch = table_batch.filter(
        lambda val, id_, md: val.sum() > 0,
        axis='observation', inplace=False
    )
    
    # Salva batch BIOM
    biom_batch = f'batch_{idx}.biom'
    with biom.util.biom_open(biom_batch, 'w') as f:
        table_batch.to_hdf5(f, f'batch_{idx}')
    
    # Lancia FAPROTAX sul batch
    out_batch = f'functional_batch_{idx}.tsv'
    cmd = [
        'python', 'collapse_table.py',
        '-i', biom_batch,
        '-o', out_batch,
        '-g', 'FAPROTAX.txt',
        '--collapse_by_metadata', 'taxonomy',
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(out_batch):
        df_batch = pd.read_csv(out_batch, sep='\t', index_col=0)
        results.append(df_batch)
        print(f'OK ({df_batch.shape})')
    else:
        print(f'ERRORE: {result.stderr[:100]}')
    
    # Rimuovi file temporanei
    os.remove(biom_batch)

# Unisci tutti i batch
print('\nUnisco i batch...')
df_functional = pd.concat(results, axis=1).fillna(0)
print('Shape finale:', df_functional.shape)

# Salva risultato finale
df_functional.to_csv('functional_table_final.tsv', sep='\t')
print('Salvato functional_table_final.tsv!')

Totale batch: 175 da 100 campioni ciascuno
Batch 1/175... OK ((92, 100))
Batch 2/175... OK ((92, 100))
Batch 3/175... OK ((92, 100))
Batch 4/175... OK ((92, 100))
Batch 5/175... OK ((92, 100))
Batch 6/175... OK ((92, 100))
Batch 7/175... OK ((92, 100))
Batch 8/175... OK ((92, 100))
Batch 9/175... OK ((92, 100))
Batch 10/175... OK ((92, 100))
Batch 11/175... OK ((92, 100))
Batch 12/175... OK ((92, 100))
Batch 13/175... OK ((92, 100))
Batch 14/175... OK ((92, 100))
Batch 15/175... OK ((92, 100))
Batch 16/175... OK ((92, 100))
Batch 17/175... OK ((92, 100))
Batch 18/175... OK ((92, 100))
Batch 19/175... OK ((92, 100))
Batch 20/175... OK ((92, 100))
Batch 21/175... OK ((92, 100))
Batch 22/175... OK ((92, 100))
Batch 23/175... OK ((92, 100))
Batch 24/175... OK ((92, 100))
Batch 25/175... OK ((92, 100))
Batch 26/175... OK ((92, 100))
Batch 27/175... OK ((92, 100))
Batch 28/175... OK ((92, 100))
Batch 29/175... OK ((92, 100))
Batch 30/175... OK ((92, 100))
Batch 31/175... OK ((92, 100))
Batch

In [2]:
from biom.table import Table
import numpy as np

# df_functional ha righe = funzioni, colonne = campioni
# Assicurati che sia nel formato giusto (funzioni × campioni)
data = df_functional.values.astype(float)
obs_ids = list(df_functional.index)    # funzioni FAPROTAX
samp_ids = list(df_functional.columns) # campioni

# Crea la BIOM table
functional_biom = Table(
    data=data,
    observation_ids=obs_ids,
    sample_ids=samp_ids
)

print('Shape BIOM funzionale:', functional_biom.shape)

# Salva
with biom.util.biom_open('functional_table_final.biom', 'w') as f:
    functional_biom.to_hdf5(f, 'faprotax_output')

print('Salvato functional_table_final.biom!')

# Verifica
table_check = biom.load_table('functional_table_final.biom')
print('Verifica shape:', table_check.shape)
print('Prime funzioni:', list(table_check.ids(axis='observation'))[:5])

Shape BIOM funzionale: (92, 17483)
Salvato functional_table_final.biom!
Verifica shape: (92, 17483)
Prime funzioni: ['methanotrophy', 'acetoclastic_methanogenesis', 'methanogenesis_by_disproportionation_of_methyl_groups', 'methanogenesis_using_formate', 'methanogenesis_by_CO2_reduction_with_H2']
